In [3]:
import pandas as pd

df = pd.read_csv("house_prices.csv")

In [4]:
def convert_amount(value):
    if value == "Call for Price":
        return None

    if "Cr" in value:
        number = float(value.replace(" Cr", ""))
        return number * 10_000_000

    if "Lac" in value:
        number = float(value.replace(" Lac", ""))
        return number * 100_000

    return None


df["Amount_numeric"] = df["Amount(in rupees)"].apply(convert_amount)

In [5]:
df[["Amount(in rupees)", "Amount_numeric"]].head(10)

,Amount(in rupees),Amount_numeric
0,42 Lac,4200000.0
1,98 Lac,9800000.0
2,1.40 Cr,14000000.0
3,25 Lac,2500000.0
4,1.60 Cr,16000000.0
5,45 Lac,4500000.0
6,16.5 Lac,1650000.0
7,60 Lac,6000000.0
8,60 Lac,6000000.0
9,1.60 Cr,16000000.0


In [6]:
df["Amount_numeric"].isnull().sum()

np.int64(9684)

In [10]:
df["Amount_numeric"].describe()
df[["Amount(in rupees)", "Amount_numeric"]].sort_values(
    "Amount_numeric", ascending=False
).head(20)
df.loc[181234, [
    "Amount(in rupees)",
    "location",
    "Carpet Area",
    "Super Area",
    "Bathroom",
    "Car Parking",
    "Transaction",
    "Ownership"
]]

Amount(in rupees)     1400.30 Cr 
location                 vadodara
Carpet Area             1252 sqft
Super Area                    NaN
Bathroom                        3
Car Parking             1 Covered
Transaction          New Property
Ownership                     NaN
Name: 181234, dtype: str

In [11]:
Q1 = df["Amount_numeric"].quantile(0.25)
Q3 = df["Amount_numeric"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Upper Bound:", upper_bound)

Q1: 4840000.0
Q3: 14500000.0
IQR: 9660000.0
Upper Bound: 28990000.0


In [12]:
outliers = df[df["Amount_numeric"] > upper_bound]

print("Number of outliers:", len(outliers))
print("Percentage:", len(outliers) / df["Amount_numeric"].notna().sum() * 100)

Number of outliers: 12283
Percentage: 6.9064982822313565


In [14]:
outliers["Amount_numeric"].describe()
df["Amount_numeric"].quantile([0.90, 0.95, 0.99, 0.995, 0.999])

0.900     24000000.0
0.950     33900000.0
0.990     70000000.0
0.995     89600000.0
0.999    125000000.0
Name: Amount_numeric, dtype: float64

In [15]:
extreme = df[df["Amount_numeric"] > 125_000_000]

print("Number of extreme values:", len(extreme))
print("Percentage:", len(extreme) / df["Amount_numeric"].notna().sum() * 100)

Number of extreme values: 174
Percentage: 0.09783690475521094


In [16]:
df_clean = df[
    (df["Amount_numeric"].notna()) &
    (df["Amount_numeric"] <= 125_000_000)
].copy()

In [17]:
print("Original shape:", df.shape)
print("Clean shape:", df_clean.shape)

Original shape: (187531, 22)
Clean shape: (177673, 22)


In [18]:
df_clean["Carpet Area"].str.extract(r'([A-Za-z]+)')[0].value_counts()

0
sqft      95884
sqyrd      4654
sqm         870
acre          2
marla         2
kanal         2
ground        1
bigha         1
Name: count, dtype: int64

In [19]:
df_clean[
    df_clean["Carpet Area"].str.contains(
        "acre|marla|kanal|ground|bigha",
        case=False,
        na=False
    )
][["Carpet Area", "location"]]

,Carpet Area,location
93644,1500 acre,kolkata
113787,2 acre,lucknow
147912,3 ground,agra
162057,1607 bigha,greater-noida
173555,14 marla,panchkula
173633,1 kanal,panchkula
173685,14 marla,panchkula
173757,1 kanal,panchkula


In [20]:
df_clean["Carpet Area"].head(20)


0      500 sqft
1      473 sqft
2      779 sqft
3      530 sqft
4      635 sqft
5           NaN
6      550 sqft
7           NaN
8           NaN
9      900 sqft
10     950 sqft
11          NaN
12          NaN
13    1820 sqft
14          NaN
15     675 sqft
16     647 sqft
17          NaN
18     600 sqft
19          NaN
Name: Carpet Area, dtype: str

In [25]:
def convert_area_to_sqft(value):
    if pd.isna(value):
        return np.nan

    value = value.lower().strip()

    number = float(value.split()[0])
    unit = value.split()[1]

    if unit == "sqft":
        return number

    elif unit == "sqyrd":
        return number * 9

    elif unit == "sqm":
        return number * 10.7639

    elif unit == "acre":
        return number * 43560

    else:
        return np.nan


df_clean["Carpet_Area_sqft"] = df_clean["Carpet Area"].apply(
    convert_area_to_sqft
)

In [26]:
df_clean[["Carpet Area", "Carpet_Area_sqft"]].head(20)

,Carpet Area,Carpet_Area_sqft
0,500 sqft,500.0
1,473 sqft,473.0
2,779 sqft,779.0
3,530 sqft,530.0
4,635 sqft,635.0
5,NaN,NaN
6,550 sqft,550.0
7,NaN,NaN
8,NaN,NaN
9,900 sqft,900.0


In [27]:
df_clean["Carpet_Area_sqft"].describe()

count    1.014100e+05
mean     1.932034e+03
std      2.052019e+05
min      1.000000e+00
25%      8.500000e+02
50%      1.100000e+03
75%      1.550000e+03
max      6.534000e+07
Name: Carpet_Area_sqft, dtype: float64

In [28]:
df_clean.nlargest(20, "Carpet_Area_sqft")[
    ["Carpet Area", "Carpet_Area_sqft", "location", "Amount(in rupees)"]
]

,Carpet Area,Carpet_Area_sqft,location,Amount(in rupees)
93644,1500 acre,6.534000e+07,kolkata,55 Lac
165733,709222 sqft,7.092220e+05,guwahati,60 Lac
149239,495970 sqft,4.959700e+05,bhiwadi,19 Lac
147580,282004 sqft,2.820040e+05,agra,20 Lac
50895,194936 sqft,1.949360e+05,gurgaon,2.60 Cr
147791,113134 sqft,1.131340e+05,agra,56 Lac
82876,107806 sqft,1.078060e+05,jaipur,68 Lac
113787,2 acre,8.712000e+04,lucknow,1.80 Cr
180072,81845 sqft,8.184500e+04,thrissur,45 Lac
176689,81675 sqft,8.167500e+04,siliguri,45.7 Lac


In [31]:
df_clean = df_clean[
    (df_clean["Carpet_Area_sqft"].isna()) |
    (df_clean["Carpet_Area_sqft"] <= 20_000)
].copy()
df_clean.shape
df_clean["Carpet_Area_sqft"].describe()

count    101383.000000
mean       1258.854558
std         737.099040
min           1.000000
25%         850.000000
50%        1100.000000
75%        1550.000000
max       18566.000000
Name: Carpet_Area_sqft, dtype: float64

In [ ]:
df_clean["Super Area"].str.extract(r'([A-Za-z]+)')[0].value_counts()


0
sqft        71847
sqyrd        3449
sqm           854
marla           8
ground          3
kanal           3
aankadam        1
acre            1
cent            1
Name: count, dtype: int64

In [36]:
def convert_super_area_to_sqft(value):
    if pd.isna(value):
        return np.nan

    value = value.lower().strip()

    number = float(value.split()[0].replace(",", ""))
    unit = value.split()[1]

    if unit == "sqft":
        return number

    elif unit == "sqyrd":
        return number * 9

    elif unit == "sqm":
        return number * 10.7639

    elif unit == "acre":
        return number * 43560

    else:
        return np.nan

In [38]:
df_clean["Super_Area_sqft"] = df_clean["Super Area"].apply(
    convert_super_area_to_sqft
)
df_clean[["Super Area", "Super_Area_sqft"]].head(20)

,Super Area,Super_Area_sqft
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
5,680 sqft,680.0
6,NaN,NaN
7,575 sqft,575.0
8,600 sqft,600.0
9,NaN,NaN


In [40]:
df_clean["Super_Area_sqft"].describe()
df_clean.nlargest(20, "Super_Area_sqft")[
    ["Super Area", "Super_Area_sqft", "location", "Amount(in rupees)"]
]

,Super Area,Super_Area_sqft,location,Amount(in rupees)
175044,998 acre,43472880.00,raipur,27 Lac
175087,"37,952 sqft",37952.00,raipur,8 Lac
17562,"36,000 sqft",36000.00,bangalore,12 Cr
71001,"25,000 sqft",25000.00,hyderabad,12 Cr
71491,"20,000 sqft",20000.00,hyderabad,10 Cr
176489,"18,000 sqft",18000.00,siliguri,75 Lac
154248,1300 sqm,13993.07,ernakulam,55 Lac
70340,"13,500 sqft",13500.00,hyderabad,3.50 Cr
71314,"13,500 sqft",13500.00,hyderabad,3.50 Cr
165788,"12,600 sqft",12600.00,guwahati,62 Lac


In [45]:
df_clean = df_clean[
    (df_clean["Super_Area_sqft"].isna()) |
    (df_clean["Super_Area_sqft"] <= 20_000)
].copy()
df_clean.shape
df_clean["Super_Area_sqft"].describe()

count    76147.000000
mean      1373.964322
std        648.037232
min          1.000000
25%       1005.000000
50%       1288.000000
75%       1675.000000
max      20000.000000
Name: Super_Area_sqft, dtype: float64

In [ ]:
df_clean["Bathroom"].value_counts()
df_clean["Bathroom"] = pd.to_numeric(
    df_clean["Bathroom"].replace("> 10", np.nan)
    
)

df_clean["Bathroom"].value_counts().sort_index()


Balcony
2       50284
1       45122
3       24923
4        9238
5         780
6         118
7          13
10         10
8          10
> 10        7
9           2
Name: count, dtype: int64

In [55]:
df_clean["Balcony"].value_counts()
df_clean["Balcony"] = pd.to_numeric(
    df_clean["Balcony"].replace("> 10", np.nan)
    
)

df_clean["Balcony"].value_counts().sort_index()


Balcony
1.0     45122
2.0     50284
3.0     24923
4.0      9238
5.0       780
6.0       118
7.0        13
8.0        10
9.0         2
10.0       10
Name: count, dtype: int64

In [56]:
df_clean["Car Parking"].value_counts().head(30)

Car Parking
1 Covered      37752
1 Covered,     16886
2 Covered       9998
1 Open          7695
2 Covered,      3957
2 Open          2560
10 Open          842
34 Covered       573
3 Covered        360
402 Covered      317
3 Covered,       122
4 Covered         94
5 Covered         32
3 Open            32
4 Open            24
15 Covered        24
5 Open            21
12 Covered        21
6 Covered         20
8 Covered         20
10 Covered        19
4 Covered,        12
20 Covered        10
7 Covered         10
50 Open            8
101 Covered        8
103 Covered        8
8 Open             8
401 Covered        7
201 Covered        7
Name: count, dtype: int64

In [58]:
df_clean["Car Parking"].dropna().str.extract(r"(\d+)")[0].astype(int).nlargest(20)
parking_num = pd.to_numeric(
    df_clean["Car Parking"].str.extract(r"(\d+)")[0],
    errors="coerce"
)

df_clean.loc[
    parking_num.nlargest(20).index,
    ["Car Parking", "location", "Amount(in rupees)", "Carpet_Area_sqft", "Super_Area_sqft"]
]

,Car Parking,location,Amount(in rupees),Carpet_Area_sqft,Super_Area_sqft
83932,999 Covered,jaipur,40 Lac,760.0,NaN
160316,908 Covered,greater-noida,69.5 Lac,NaN,1250.0
52152,903 Covered,gurgaon,96 Lac,1270.0,NaN
166051,901 Covered,gwalior,55 Lac,1250.0,NaN
155338,835 Covered,faridabad,40 Lac,NaN,1220.0
82443,818 Open,jaipur,12 Lac,750.0,NaN
160317,808 Open,greater-noida,15 Lac,425.0,NaN
148020,804 Open,allahabad,45 Lac,NaN,1250.0
145278,801 Covered,pune,57 Lac,NaN,1050.0
169875,801 Covered,mangalore,80 Lac,1100.0,NaN


In [ ]:
parking_num = pd.to_numeric(
    df_clean["Car Parking"].str.extract(r"(\d+)")[0],
    errors="coerce"
)

parking_num = parking_num.where(parking_num <= 10, np.nan)

df_clean["Car_Parking_Count"] = parking_num

df_clean["Car_Parking_Count"].value_counts().sort_index()

df_clean[["Car Parking", "Car_Parking_Count"]].dropna().head(20)

df_clean.drop(columns=["Car Parking"], inplace=True)


In [66]:
df_clean["Floor"].value_counts().head(30)

Floor
1 out of 4          11796
2 out of 4          11425
3 out of 4           7789
1 out of 3           6929
2 out of 3           5868
4 out of 4           5813
3 out of 3           4398
2 out of 5           4329
4 out of 5           3723
3 out of 6           3578
3 out of 5           3355
2 out of 2           3349
1 out of 5           3314
1 out of 2           2829
Ground out of 4      2766
5 out of 5           2404
8 out of 10          2380
Ground out of 10     2154
3 out of 10          1967
4 out of 8           1888
6 out of 8           1867
5 out of 10          1757
Ground out of 2      1736
1 out of 7           1662
5 out of 7           1624
1 out of 6           1512
Ground out of 5      1365
Ground out of 3      1302
6 out of 7           1286
2 out of 6           1237
Name: count, dtype: int64

In [68]:
def extract_floor(value):
    if pd.isna(value):
        return np.nan, np.nan

    value = value.strip()

    # تقسيم القيمة
    parts = value.split(" out of ")

    if len(parts) != 2:
        return np.nan, np.nan

    current = parts[0].strip()
    total = parts[1].strip()

    # إجمالي الأدوار
    try:
        total_floors = float(total)
    except:
        total_floors = np.nan

    # الدور الحالي
    if current == "Ground":
        current_floor = 0

    elif current == "Upper Basement":
        current_floor = -1

    elif current == "Lower Basement":
        current_floor = -2

    else:
        try:
            current_floor = float(current)
        except:
            current_floor = np.nan

    return current_floor, total_floors


df_clean[["Current_Floor", "Total_Floors"]] = df_clean["Floor"].apply(
    lambda x: pd.Series(extract_floor(x))
)

In [ ]:
df_clean[["Floor", "Current_Floor", "Total_Floors"]].head(20)

df_clean[["Current_Floor", "Total_Floors"]].describe()

df_clean[
    df_clean["Current_Floor"] > df_clean["Total_Floors"]
][["Floor", "Current_Floor", "Total_Floors"]].head(20)

,Floor,Current_Floor,Total_Floors
72673,6 out of 2,6.0,2.0
113564,4 out of 2,4.0,2.0
158590,4 out of 3,4.0,3.0


In [ ]:
invalid_floor = (
    df_clean["Current_Floor"] > df_clean["Total_Floors"]
)

df_clean.loc[
    invalid_floor,
    ["Current_Floor", "Total_Floors"]
] = np.nan

df_clean[
    df_clean["Current_Floor"] > df_clean["Total_Floors"]
][["Floor", "Current_Floor", "Total_Floors"]]

df_clean.drop(columns=["Floor"], inplace=True)

df_clean.shape

,Floor,Current_Floor,Total_Floors


In [ ]:

df_clean["Ownership"].value_counts(dropna=False)

Ownership
Freehold                106236
NaN                      61902
Leasehold                 5148
Co-operative Society      3344
Power Of Attorney         1012
Name: count, dtype: int64

In [78]:
df_clean["Transaction"].value_counts(dropna=False)

Transaction
Resale          135506
New Property     41364
Other              703
NaN                 67
Rent/Lease           2
Name: count, dtype: int64

In [79]:
df_clean["Furnishing"].value_counts(dropna=False)

Furnishing
Semi-Furnished    82760
Unfurnished       73418
Furnished         19400
NaN                2064
Name: count, dtype: int64

In [80]:
df_clean["facing"].value_counts(dropna=False)

facing
NaN             65704
East            52171
North - East    23293
North           15282
West             8466
South            4311
North - West     3801
South - East     2565
South -West      2049
Name: count, dtype: int64

In [ ]:
df_clean["overlooking"].value_counts(dropna=False)

df_clean["overlooking"].dropna().unique()

df_clean["Has_Main_Road"] = df_clean["overlooking"].str.contains(
    "Main Road", na=False
).astype(int)

df_clean["Has_Garden_Park"] = df_clean["overlooking"].str.contains(
    "Garden/Park", na=False
).astype(int)

df_clean["Has_Pool"] = df_clean["overlooking"].str.contains(
    "Pool", na=False
).astype(int)
df_clean[
    ["overlooking", "Has_Main_Road", "Has_Garden_Park", "Has_Pool"]
].head(20)



,overlooking,Has_Main_Road,Has_Garden_Park,Has_Pool
0,NaN,0,0,0
1,Garden/Park,0,1,0
2,Garden/Park,0,1,0
3,NaN,0,0,0
4,"Garden/Park, Main Road",1,1,0
5,"Garden/Park, Main Road",1,1,0
6,NaN,0,0,0
7,NaN,0,0,0
8,NaN,0,0,0
9,Garden/Park,0,1,0


In [ ]:
df_clean.drop(columns=["overlooking"], inplace=True)

df_clean.shape

(177642, 27)

In [90]:
df_clean["location"].value_counts(dropna=False).head(20)

df_clean["location"].value_counts().tail(20)

location
trichy         119
udaipur        119
bhopal         118
rajahmundry     90
tirupati        90
bhiwandi        88
belgaum         60
jodhpur         60
udupi           60
kozhikode       59
satara          59
shimla          59
vrindavan       59
ahmadnagar      30
navsari         30
palakkad        30
solapur         30
nellore         29
pondicherry     29
madurai         26
Name: count, dtype: int64

In [93]:
df_clean["Society"].value_counts(dropna=False).head(20)

df_clean["Society"].nunique(dropna=True)
df_clean["Society"].nunique(dropna=False)


10041

In [96]:
society_freq = df_clean["Society"].value_counts()

df_clean["Society_Frequency"] = (
    df_clean["Society"]
    .map(society_freq)
    .fillna(0)
)
df_clean[["Society", "Society_Frequency"]].head(20)

,Society,Society_Frequency
0,Srushti Siddhi Mangal Murti Complex,1.0
1,Dosti Vihar,1.0
2,Sunrise by Kalpataru,6.0
3,NaN,0.0
4,TenX Habitat Raymond Realty,6.0
5,Virat Aangan,2.0
6,NaN,0.0
7,NaN,0.0
8,NaN,0.0
9,Pride Palms,4.0


In [ ]:
df_clean.drop(columns=["Society"], inplace=True)
df_clean.shape

(177642, 27)

In [100]:
df_clean.info()

df_clean.drop(
    columns=["Index", "Dimensions", "Plot Area"],
    inplace=True
)

<class 'pandas.DataFrame'>
Index: 177642 entries, 0 to 187530
Data columns (total 27 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Index              177642 non-null  int64  
 1   Title              177642 non-null  str    
 2   Description        174714 non-null  str    
 3   Amount(in rupees)  177642 non-null  str    
 4   Price (in rupees)  169666 non-null  float64
 5   location           177642 non-null  str    
 6   Carpet Area        101389 non-null  str    
 7   Status             177047 non-null  str    
 8   Transaction        177575 non-null  str    
 9   Furnishing         175578 non-null  str    
 10  facing             111938 non-null  str    
 11  Bathroom           176863 non-null  float64
 12  Balcony            130500 non-null  float64
 13  Ownership          115740 non-null  str    
 14  Super Area         76163 non-null   str    
 15  Dimensions         0 non-null       float64
 16  Plot Area         

In [103]:
df_clean[["Price (in rupees)", "Amount_numeric"]].describe()
df_clean[["Price (in rupees)", "Amount_numeric"]].corr()

,Price (in rupees),Amount_numeric
Price (in rupees),1.000000,0.215899
Amount_numeric,0.215899,1.000000


In [ ]:
df_clean["Title"].value_counts().head(20)

df_clean.drop(
    columns=["Title", "Description"],
    inplace=True
)

df_clean.shape

(177642, 22)

In [112]:
df_clean[
    ["Amount(in rupees)", "Price (in rupees)", "Amount_numeric"]
].dropna().head(10)

,Amount(in rupees),Price (in rupees),Amount_numeric
0,42 Lac,6000.0,4200000.0
1,98 Lac,13799.0,9800000.0
2,1.40 Cr,17500.0,14000000.0
4,1.60 Cr,18824.0,16000000.0
5,45 Lac,6618.0,4500000.0
6,16.5 Lac,2538.0,1650000.0
7,60 Lac,10435.0,6000000.0
8,60 Lac,10000.0,6000000.0
9,1.60 Cr,11150.0,16000000.0
10,1.40 Cr,12174.0,14000000.0


In [113]:
df_clean.drop(
    columns=[
        "Amount(in rupees)",
        "Price (in rupees)",
        "Carpet Area",
        "Super Area"
    ],
    inplace=True
)

In [114]:
df_clean.shape

(177642, 18)

In [115]:
df_clean.to_csv("cleaned_house_prices.csv", index=False)

In [116]:
import os

print(os.path.exists("cleaned_house_prices.csv"))

True
